In [2]:
# Google Colab setup: fetch this repository and use this notebook's directory.
!pip install boto3
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    !git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_21'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 4.1 MB/s eta 0:00:00
Cloning into '/content/BITS_programming'...
remote: Enumerating objects: 2038, done.
remote: Counting objects: 100% (694/694), done.
remote: Compressing objects: 100% (454/454), done.
remote: Total 2038 (delta 236), reused 561 (delta 147), pack-reused 1344 (from 3)
Receiving objects: 100% (2038/2038), 269.04 MiB | 37.65 MiB/s, done.
Resolving deltas: 100% (300/300), done.
Working directory: /content/BITS_programming/assignments/assignment_21


In [3]:
import os

def get_colab_secret(*names, required=True):
    """Read the first available credential from env vars or Colab Secrets."""
    # Environment variables make the notebook work in Colab, local Jupyter,
    # and managed notebook runtimes without changing the code.
    for name in names:
        value = os.environ.get(name)
        if value:
            return value

    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    if userdata is not None:
        for name in names:
            try:
                value = userdata.get(name)
                if value:
                    return value
            except Exception:
                # Colab may deny a secret that was not shared with this notebook.
                continue

    if required:
        choices = " or ".join(names)
        raise RuntimeError(
            f"Missing AWS credential. Add a Colab Secret named {choices} and enable notebook access, "
            f"or set one of those environment variables."
        )
    return None

# Fetch AWS credentials from Colab secrets
AWS_ACCESS_KEY_ID = get_colab_secret('AWS_ACCESS_KEY_ID', 'AWS_ACCESS_KEY')
AWS_SECRET_ACCESS_KEY = get_colab_secret('AWS_SECRET_ACCESS_KEY', 'AWS_SECRET_KEY')
AWS_SESSION_TOKEN = get_colab_secret('AWS_SESSION_TOKEN', 'AWS_SECURITY_TOKEN', required=False)

os.environ['AWS_ACCESS_KEY_ID'] = AWS_ACCESS_KEY_ID
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET_ACCESS_KEY
if AWS_SESSION_TOKEN:
    os.environ['AWS_SESSION_TOKEN'] = AWS_SESSION_TOKEN

print("AWS credentials loaded from Colab secrets and set as environment variables.")

AWS credentials loaded from Colab secrets and set as environment variables.


In [4]:
# Install missing packages
!pip install onnx onnxruntime sagemaker

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which 

# Lesson 6 · Model Optimization I — Quantization & Acceleration
### Real AWS Implementation via SageMaker Neo

**Module 05 · AI Platform Engineering · BITS Pilani**

This notebook implements the Lesson 6 concepts against **real AWS services** (no mocks, no local-only pipelines).
The centrepiece is **AWS SageMaker Neo** — the managed model-compilation service that applies INT8 quantization,
graph optimization, and hardware-specific kernel generation in one API call.

## What you'll build

| Step | Action | AWS service used |
|------|--------|------------------|
| 1 | Preflight (imports, region, IAM) | STS, boto3 |
| 2 | Config with **auto-detected PyTorch version** | none |
| 3 | Train a small classifier locally (works around training-quota=0) | none — pure Python |
| 4 | Export to ONNX + dynamic INT8 quantization | onnxruntime (local) |
| 5 | Benchmark FP32 vs ONNX vs ONNX-INT8 (accuracy, latency, size) | none — pure Python |
| 6 | Package + upload PyTorch model to S3 | **S3** |
| 7 | Launch SageMaker Neo compilation job (target = `ml_m5`) | **SageMaker Neo** |
| 8 | Deploy compiled + baseline models to two endpoints | **SageMaker Endpoint** |
| 9 | Smoke test + load test both endpoints | **SageMaker Runtime** |
| 10 | Compare p50/p95 latency + throughput + cost per 1k requests | pandas |
| 11 | Guarded teardown | SageMaker, S3, IAM |

## Cost + time budget

| Resource | Rate | Runtime | Cost |
|---|---|---|---|
| Neo compilation job | included in SageMaker | ~5–8 min | $0.00 |
| 2× ml.m5.large endpoints | $0.14/hr each | ~15 min | ~$0.07 |
| S3 storage | ~$0.023/GB/mo | model artifacts | ~$0.001 |
| **Total** | | **~15 min** | **~$0.08** |

## Prerequisites

- SageMaker Studio JupyterLab in **eu-north-1** (or any region where you have `ml.m5.large` endpoint quota ≥ 2)
- IAM role attached: `AmazonSageMakerAdminIAMExecutionRole` (or equivalent with SageMaker + S3 + IAM read access)
- Kernel: `Python 3` on `SageMaker Distribution 3.x` (has PyTorch 2.x pre-installed)


## 1. Preflight — verify environment before spending AWS budget

Read-only. Confirms package versions, IAM identity, region. Halts with a clear message if anything
is missing so you don't waste time on Section 6+.

In [5]:
# Block 0 - Preflight (read-only, no AWS mutations)
import sys, os, importlib
import boto3
from botocore.exceptions import ClientError

_ok = True

def _check_pkg(name, submodule=None):
    """Check that a package (and optional submodule) can be imported."""
    global _ok
    try:
        mod = importlib.import_module(name)
        v = getattr(mod, "__version__", "?")
        if submodule:
            importlib.import_module(f"{name}.{submodule}")
            print(f"  PASS  {name:24s} {v}  (with .{submodule})")
        else:
            print(f"  PASS  {name:24s} {v}")
        return True
    except ImportError as e:
        print(f"  FAIL  {name}{'.' + submodule if submodule else ''}: {e}")
        _ok = False
        return False

print("=== Package versions ===")
_check_pkg("boto3")
_check_pkg("numpy")
_check_pkg("sklearn")
_check_pkg("torch")
_check_pkg("onnx")
_check_pkg("onnxruntime", submodule="quantization")   # need the quantization submodule
_check_pkg("sagemaker")

print("\n=== AWS identity + region ===")
sts = boto3.client("sts")
try:
    ident = sts.get_caller_identity()
    print(f"  PASS  identity    {ident['Arn']}")
    print(f"  PASS  account_id  {ident['Account']}")
except ClientError as e:
    print(f"  FAIL  cannot get caller identity: {e}")
    _ok = False

_session = boto3.Session()
_region = _session.region_name or os.environ.get("AWS_DEFAULT_REGION")
if _region:
    print(f"  PASS  region      {_region}")
else:
    print(f"  WARN  no default region set. Block 1 will force eu-north-1.")

if not _ok:
    raise RuntimeError("Preflight FAILED - fix rows marked FAIL before proceeding.")
print("\nPreflight passed. Safe to proceed.")

=== Package versions ===
  PASS  boto3                    1.43.92
  PASS  numpy                    2.1.3
  PASS  sklearn                  1.6.1
  PASS  torch                    2.11.0+cpu
  PASS  onnx                     1.22.0
  PASS  onnxruntime              1.30.0  (with .quantization)
  PASS  sagemaker                ?

=== AWS identity + region ===
  PASS  identity    arn:aws:iam::061831608851:root
  PASS  account_id  061831608851
  WARN  no default region set. Block 1 will force eu-north-1.

Preflight passed. Safe to proceed.


## 2. Configuration + PyTorch version auto-detection

Single edit point at the top: `AWS_REGION_NAME` and `ALLOW_AWS_MUTATIONS`.
PyTorch container + Neo framework versions are **derived from your kernel's torch version**
so tracing format matches what Neo expects.

In [6]:
import os

def get_colab_secret(*names, required=True):
    """Read the first available credential from env vars or Colab Secrets."""
    # Environment variables make the notebook work in Colab, local Jupyter,
    # and managed notebook runtimes without changing the code.
    for name in names:
        value = os.environ.get(name)
        if value:
            return value

    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    if userdata is not None:
        for name in names:
            try:
                value = userdata.get(name)
                if value:
                    return value
            except Exception:
                # Colab may deny a secret that was not shared with this notebook.
                continue

    if required:
        choices = " or ".join(names)
        raise RuntimeError(
            f"Missing AWS credential. Add a Colab Secret named {choices} and enable notebook access, "
            f"or set one of those environment variables."
        )
    return None

# Block 1 - Configuration + version detection
import json
import torch, datetime as dt
from pathlib import Path

# --------------------------------------------------------------------
# USER-EDITABLE
# --------------------------------------------------------------------
AWS_REGION_NAME       = "eu-north-1"
ALLOW_AWS_MUTATIONS   = True

PROJECT_NAME          = "lesson6-neo"
ENVIRONMENT           = "demo"
NEO_TARGET_DEVICE     = "ml_m5"      # target CPU family for Neo compilation
ENDPOINT_INSTANCE     = "ml.m5.large"

# --------------------------------------------------------------------
# AUTO-DETECTED - kernel PyTorch version drives everything downstream
# --------------------------------------------------------------------
_full_torch_ver = torch.__version__.split("+")[0]             # e.g. "2.8.0" from "2.8.0+cu121"
_torch_mm       = ".".join(_full_torch_ver.split(".")[:2])    # e.g. "2.8"

# Map kernel torch version to SageMaker container versions.
# Recent torch (2.4+) still maps to pytorch container 2.1 - AWS DLCs lag torch by ~2 minor.
_VERSION_MAP = {
    "2.9": {"pytorch_img": "2.1", "py_img": "py310", "neo_fw": "2.0"},
    "2.8": {"pytorch_img": "2.1", "py_img": "py310", "neo_fw": "2.0"},
    "2.7": {"pytorch_img": "2.1", "py_img": "py310", "neo_fw": "2.0"},
    "2.6": {"pytorch_img": "2.1", "py_img": "py310", "neo_fw": "2.0"},
    "2.5": {"pytorch_img": "2.1", "py_img": "py310", "neo_fw": "2.0"},
    "2.4": {"pytorch_img": "2.1", "py_img": "py310", "neo_fw": "2.0"},
    "2.3": {"pytorch_img": "2.1", "py_img": "py310", "neo_fw": "2.0"},
    "2.2": {"pytorch_img": "2.1", "py_img": "py310", "neo_fw": "2.0"},
    "2.1": {"pytorch_img": "2.1", "py_img": "py310", "neo_fw": "2.0"},
    "2.0": {"pytorch_img": "2.0", "py_img": "py310", "neo_fw": "2.0"},
    "1.13":{"pytorch_img": "1.13","py_img": "py39",  "neo_fw": "1.13"},
}
_versions = _VERSION_MAP.get(_torch_mm, {
    "pytorch_img": "2.1", "py_img": "py310", "neo_fw": "2.0"  # fallback for future versions
})
PYTORCH_IMG_VER  = _versions["pytorch_img"]
PY_IMG_VER       = _versions["py_img"]
NEO_FRAMEWORK    = "PYTORCH"
NEO_FW_VERSION   = _versions["neo_fw"]

print(f"Detected kernel PyTorch: {_full_torch_ver}")
print(f"Using PyTorch container: {PYTORCH_IMG_VER} / {PY_IMG_VER}")
print(f"Using Neo framework:     {NEO_FRAMEWORK} {NEO_FW_VERSION}")

# Fetch AWS credentials from Colab secrets
AWS_ACCESS_KEY_ID = get_colab_secret('AWS_ACCESS_KEY_ID', 'AWS_ACCESS_KEY')
AWS_SECRET_ACCESS_KEY = get_colab_secret('AWS_SECRET_ACCESS_KEY', 'AWS_SECRET_KEY')
AWS_SESSION_TOKEN = get_colab_secret('AWS_SESSION_TOKEN', 'AWS_SECURITY_TOKEN', required=False)

os.environ['AWS_ACCESS_KEY_ID'] = AWS_ACCESS_KEY_ID
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET_ACCESS_KEY
if AWS_SESSION_TOKEN:
    os.environ['AWS_SESSION_TOKEN'] = AWS_SESSION_TOKEN

print("AWS credentials loaded from Colab secrets and set as environment variables.")

# --------------------------------------------------------------------
# Region + clients
# --------------------------------------------------------------------
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION_NAME
os.environ["AWS_REGION"]         = AWS_REGION_NAME

boto_session  = boto3.Session(region_name=AWS_REGION_NAME)
region        = AWS_REGION_NAME
s3            = boto_session.client("s3")
sm            = boto_session.client("sagemaker")
runtime       = boto_session.client("sagemaker-runtime")
iam           = boto_session.client("iam")

_ident     = sts.get_caller_identity()
account_id = _ident["Account"]
caller_arn = _ident["Arn"]

# Deterministic + unique-per-run resource names
timestamp = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
bucket_name         = f"{PROJECT_NAME}-{account_id}-{region}"
prefix              = f"{PROJECT_NAME}/{ENVIRONMENT}"
model_s3_prefix     = f"{prefix}/models"
neo_output_prefix   = f"{prefix}/neo-output"

compilation_job     = f"{PROJECT_NAME}-{ENVIRONMENT}-compile-{timestamp}"
model_name_baseline = f"{PROJECT_NAME}-{ENVIRONMENT}-baseline-{timestamp}"
model_name_neo      = f"{PROJECT_NAME}-{ENVIRONMENT}-neo-{timestamp}"
endpoint_baseline   = f"{PROJECT_NAME}-{ENVIRONMENT}-baseline-ep"
endpoint_neo        = f"{PROJECT_NAME}-{ENVIRONMENT}-neo-ep"
config_baseline     = f"{endpoint_baseline}-cfg-{timestamp}"
config_neo          = f"{endpoint_neo}-cfg-{timestamp}"

# Execution role - use the SageMaker default, with fallback
try:
    from sagemaker import get_execution_role
    execution_role = get_execution_role()
except Exception as _e:
    # Fallback: derive from the caller's assumed role
    if ":assumed-role/" in caller_arn:
        _role_name = caller_arn.split("/")[1]
        execution_role = f"arn:aws:iam::{account_id}:role/{_role_name}"
    else:
        execution_role = caller_arn
    print(f"Note: sagemaker.get_execution_role() unavailable ({type(_e).__name__}); using {execution_role}")

work_dir = Path("./neo_work")
work_dir.mkdir(exist_ok=True)

print(f"\nRegion:          {region}")
print(f"Account:         {account_id}")
print(f"Execution role:  {execution_role}")
print(f"S3 bucket:       {bucket_name}")
print(f"Neo target:      {NEO_TARGET_DEVICE}")
print(f"Endpoints:       {endpoint_baseline} + {endpoint_neo}")


Detected kernel PyTorch: 2.11.0
Using PyTorch container: 2.1 / py310
Using Neo framework:     PYTORCH 2.0
AWS credentials loaded from Colab secrets and set as environment variables.
Note: sagemaker.get_execution_role() unavailable (ImportError); using arn:aws:iam::061831608851:root

Region:          eu-north-1
Account:         061831608851
Execution role:  arn:aws:iam::061831608851:root
S3 bucket:       lesson6-neo-061831608851-eu-north-1
Neo target:      ml_m5
Endpoints:       lesson6-neo-demo-baseline-ep + lesson6-neo-demo-neo-ep


/tmp/ipykernel_1206/3866095492.py:114: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")


## 3. Ensure S3 bucket exists

Idempotent: creates on first run, no-op on subsequent runs.

In [7]:
# Block 2 - Ensure S3 bucket exists (idempotent)
if ALLOW_AWS_MUTATIONS:
    try:
        if region == "us-east-2":
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={"LocationConstraint": region},
            )
        print(f"Created bucket: s3://{bucket_name}")
    except ClientError as e:
        code_ = e.response["Error"]["Code"]
        if code_ in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
            print(f"Bucket already exists: s3://{bucket_name}")
        else:
            raise
else:
    print("Skipped (ALLOW_AWS_MUTATIONS=False).")


Bucket already exists: s3://lesson6-neo-061831608851-eu-north-1


## 4. Train a small classifier locally

**Why local, not SageMaker Training?** Our sandbox has `training job usage = 0 instances` for every ml.* type.
Neo compilation is a *separate* quota bucket that works, so we train in the kernel and hand a ready model
to Neo. In a production account with training quota, you'd use `sagemaker.pytorch.PyTorch` as the estimator
and skip this section.

The model is intentionally tiny — a 2-layer MLP on `make_classification(n_features=20)` — so training
finishes in seconds. The point of the lesson is what happens **after** training.

In [8]:
# Block 3 - Train a small MLP locally (PyTorch)
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Reproducible synthetic dataset
X, y = make_classification(
    n_samples=5000, n_features=20, n_informative=15,
    n_redundant=3, n_classes=2, random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)

# Cast to float32 (Neo cares about explicit dtypes)
X_train = X_train.astype(np.float32)
X_test  = X_test.astype(np.float32)
y_train = y_train.astype(np.int64)
y_test  = y_test.astype(np.int64)

class TabularMLP(nn.Module):
    """Small classifier - kept intentionally simple so Neo compilation is fast."""
    def __init__(self, n_features=20, n_hidden=64, n_classes=2):
        super().__init__()
        self.fc1 = nn.Linear(n_features, n_hidden)
        self.fc2 = nn.Linear(n_hidden, n_hidden)
        self.fc3 = nn.Linear(n_hidden, n_classes)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)   # logits (no softmax — endpoint applies softmax later)

torch.manual_seed(42)
model = TabularMLP()
optim_ = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)
X_test_t  = torch.from_numpy(X_test)

model.train()
for epoch in range(30):
    optim_.zero_grad()
    logits = model(X_train_t)
    loss = loss_fn(logits, y_train_t)
    loss.backward()
    optim_.step()
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            pred = model(X_test_t).argmax(dim=1).numpy()
            acc = accuracy_score(y_test, pred)
            print(f"  epoch {epoch+1:3d}  loss={loss.item():.4f}  test_acc={acc:.4f}")

model.eval()
with torch.no_grad():
    baseline_pred = model(X_test_t).argmax(dim=1).numpy()
baseline_acc = accuracy_score(y_test, baseline_pred)
baseline_f1  = f1_score(y_test, baseline_pred, average="binary")
print(f"\nBASELINE (FP32 PyTorch)  accuracy={baseline_acc:.4f}  f1={baseline_f1:.4f}")


  epoch  10  loss=0.5744  test_acc=0.7090
  epoch  20  loss=0.4654  test_acc=0.7990
  epoch  30  loss=0.3730  test_acc=0.8590

BASELINE (FP32 PyTorch)  accuracy=0.8590  f1=0.8605


## 5. Export to ONNX and apply INT8 quantization

Two things happen here:
1. **Export** the PyTorch model to ONNX — a portable IR that most quantization tools consume.
2. **Dynamic INT8 quantize** with `onnxruntime.quantization` — no calibration data needed, just weights.

This section is **local-only** and serves to teach the concept. The AWS Neo compilation later (Section 7)
consumes the PyTorch traced model directly.

In [9]:
# Block 4 - Export PyTorch -> ONNX (local benchmarking only)
import onnx
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

onnx_fp32_path = work_dir / "model_fp32.onnx"

# Dummy input shape: (batch, features). ONNX needs concrete tensor to trace.
dummy = torch.randn(1, 20, dtype=torch.float32)

torch.onnx.export(
    model,
    dummy,
    onnx_fp32_path.as_posix(),
    input_names  = ["input"],
    output_names = ["logits"],
    dynamic_axes = {"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=13,
    do_constant_folding=True,
    dynamo=False,   # force legacy TorchScript exporter (no onnxscript required)
)

onnx_model = onnx.load(onnx_fp32_path.as_posix())
onnx.checker.check_model(onnx_model)
print(f"ONNX (FP32) exported: {onnx_fp32_path}  size={onnx_fp32_path.stat().st_size:,} bytes")


ONNX (FP32) exported: neo_work/model_fp32.onnx  size=23,230 bytes


In [10]:
# Block 5 - Apply dynamic INT8 quantization (onnxruntime)
from onnxruntime.quantization import quantize_dynamic, QuantType

onnx_int8_path = work_dir / "model_int8.onnx"

quantize_dynamic(
    model_input  = onnx_fp32_path.as_posix(),
    model_output = onnx_int8_path.as_posix(),
    weight_type  = QuantType.QInt8,
)

fp32_size = onnx_fp32_path.stat().st_size
int8_size = onnx_int8_path.stat().st_size
print(f"ONNX FP32 size: {fp32_size:>8,} bytes")
print(f"ONNX INT8 size: {int8_size:>8,} bytes")
print(f"Compression:    {fp32_size / int8_size:.2f}x smaller  ({(1 - int8_size/fp32_size) * 100:.1f}% reduction)")


ONNX FP32 size:   23,230 bytes
ONNX INT8 size:    9,356 bytes
Compression:    2.48x smaller  (59.7% reduction)


## 6. Local benchmark: FP32 vs ONNX-FP32 vs ONNX-INT8

Measure the three variants **in-kernel** before touching AWS. Latency is per-request p50 over 500 calls,
batch size = 1.

In [11]:
# Block 6 - Local latency + accuracy benchmark
import time
import onnxruntime as ort
import pandas as pd

def bench(session_or_model, X, y, n_runs=500, is_torch=False):
    """Run n_runs single-sample forward passes, return metrics dict."""
    # Warm up (JIT / kernel selection)
    for i in range(20):
        if is_torch:
            with torch.no_grad():
                _ = session_or_model(torch.from_numpy(X[i:i+1]))
        else:
            _ = session_or_model.run(None, {"input": X[i:i+1]})

    latencies_ms = []
    preds = []
    for i in range(min(n_runs, len(X))):
        t0 = time.perf_counter()
        if is_torch:
            with torch.no_grad():
                logits = session_or_model(torch.from_numpy(X[i:i+1])).numpy()
        else:
            logits = session_or_model.run(None, {"input": X[i:i+1]})[0]
        latencies_ms.append((time.perf_counter() - t0) * 1000)
        preds.append(int(np.argmax(logits, axis=1)[0]))

    return {
        "p50_ms":         np.percentile(latencies_ms, 50),
        "p95_ms":         np.percentile(latencies_ms, 95),
        "p99_ms":         np.percentile(latencies_ms, 99),
        "throughput_rps": 1000.0 / np.mean(latencies_ms),
        "accuracy":       accuracy_score(y[:len(preds)], preds),
    }

# 1. PyTorch FP32 baseline
r1 = bench(model, X_test, y_test, is_torch=True)
r1["size_bytes"] = sum(p.numel() * p.element_size() for p in model.parameters())

# 2. ONNX FP32
sess_fp32 = ort.InferenceSession(onnx_fp32_path.as_posix(), providers=["CPUExecutionProvider"])
r2 = bench(sess_fp32, X_test, y_test)
r2["size_bytes"] = fp32_size

# 3. ONNX INT8 (dynamic quantized)
sess_int8 = ort.InferenceSession(onnx_int8_path.as_posix(), providers=["CPUExecutionProvider"])
r3 = bench(sess_int8, X_test, y_test)
r3["size_bytes"] = int8_size

df_local = pd.DataFrame([r1, r2, r3], index=[
    "PyTorch FP32",
    "ONNX FP32",
    "ONNX INT8 (dynamic)",
])
df_local["size_KB"] = df_local["size_bytes"] / 1024
df_local = df_local[["accuracy", "p50_ms", "p95_ms", "p99_ms", "throughput_rps", "size_KB"]]
df_local = df_local.round({"accuracy": 4, "p50_ms": 3, "p95_ms": 3, "p99_ms": 3,
                            "throughput_rps": 1, "size_KB": 2})
print("=== Local benchmark ===")
print(df_local.to_string())
print()
print(f"INT8 vs FP32 speedup (p50):  {r2['p50_ms'] / r3['p50_ms']:.2f}x")
print(f"INT8 accuracy delta:         {(r3['accuracy'] - r1['accuracy'])*100:+.2f} percentage points")


=== Local benchmark ===
                     accuracy  p50_ms  p95_ms  p99_ms  throughput_rps  size_KB
PyTorch FP32            0.852   0.072   0.146   2.242          7012.0    22.01
ONNX FP32               0.852   0.021   0.033   0.054         42539.1    22.69
ONNX INT8 (dynamic)     0.848   0.023   0.037   0.093         22947.3     9.14

INT8 vs FP32 speedup (p50):  0.92x
INT8 accuracy delta:         -0.40 percentage points


## 7. SageMaker Neo — real AWS compilation

1. Package the **PyTorch traced model** (Neo consumes TorchScript archives)
2. Upload the tarball to S3
3. Launch a Neo compilation job targeting `ml_m5`
4. Wait ~5–8 min for compilation
5. Neo writes the compiled artifact back to S3

Neo internally uses **TVM** + **LLVM** + vendor kernels. From your perspective it's a single API call.

In [12]:
# Block 7 - Package the PyTorch model for Neo
import tarfile

# Trace the model with a fixed input shape
model.eval()
example_input = torch.randn(1, 20, dtype=torch.float32)
traced = torch.jit.trace(model, example_input)
traced_path = work_dir / "model.pth"
traced.save(traced_path.as_posix())

# Package into tar.gz (Neo expects only model.pth at the root)
model_tar_path = work_dir / "model_neo_input.tar.gz"
with tarfile.open(model_tar_path.as_posix(), "w:gz") as tar:
    tar.add(traced_path.as_posix(), arcname="model.pth")

print(f"Packaged: {model_tar_path}  size={model_tar_path.stat().st_size:,} bytes")

s3_input_key = f"{model_s3_prefix}/model_neo_input.tar.gz"
s3.upload_file(model_tar_path.as_posix(), bucket_name, s3_input_key)
model_s3_uri = f"s3://{bucket_name}/{s3_input_key}"
print(f"Uploaded -> {model_s3_uri}")


Packaged: neo_work/model_neo_input.tar.gz  size=27,419 bytes
Uploaded -> s3://lesson6-neo-061831608851-eu-north-1/lesson6-neo/demo/models/model_neo_input.tar.gz


In [13]:
# Create a minimal Neo-only role - clean trust, only the policies Neo needs
import json as _json
import time as _t

neo_role_name = "SageMakerNeoLabRole"
neo_role_arn  = f"arn:aws:iam::{account_id}:role/{neo_role_name}"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

try:
    iam.create_role(
        RoleName=neo_role_name,
        AssumeRolePolicyDocument=_json.dumps(trust_policy),
        Description="Minimal role for SageMaker Neo compilation (lab)",
    )
    print(f"Created role: {neo_role_arn}")
except iam.exceptions.EntityAlreadyExistsException:
    iam.update_assume_role_policy(
        RoleName=neo_role_name, PolicyDocument=_json.dumps(trust_policy),
    )
    print(f"Role exists (trust refreshed): {neo_role_arn}")

# Attach SageMaker + S3 access
for policy_arn in [
    "arn:aws:iam::aws:policy/AmazonSageMakerFullAccess",
    "arn:aws:iam::aws:policy/AmazonS3FullAccess",
]:
    try:
        iam.attach_role_policy(RoleName=neo_role_name, PolicyArn=policy_arn)
        print(f"  Attached: {policy_arn.split('/')[-1]}")
    except iam.exceptions.LimitExceededException:
        print(f"  Skip {policy_arn}: quota (unlikely on new role)")
    except Exception as e:
        print(f"  Skip {policy_arn}: {type(e).__name__}")

print("\nWaiting 30s for IAM propagation...")
_t.sleep(30)

neo_execution_role = neo_role_arn
print(f"\nUse this for Neo: {neo_execution_role}")

Role exists (trust refreshed): arn:aws:iam::061831608851:role/SageMakerNeoLabRole
  Attached: AmazonSageMakerFullAccess
  Attached: AmazonS3FullAccess

Waiting 30s for IAM propagation...

Use this for Neo: arn:aws:iam::061831608851:role/SageMakerNeoLabRole


In [14]:
# Fresh job name + new role
timestamp = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
compilation_job = f"{PROJECT_NAME}-{ENVIRONMENT}-compile-{timestamp}"

data_input_cfg = json.dumps({"input0": [1, 20]})
neo_output_uri = f"s3://{bucket_name}/{neo_output_prefix}/"

sm.create_compilation_job(
    CompilationJobName=compilation_job,
    RoleArn=neo_execution_role,    # <-- new role, not execution_role
    InputConfig={
        "S3Uri":            model_s3_uri,
        "DataInputConfig":  data_input_cfg,
        "Framework":        NEO_FRAMEWORK,
        "FrameworkVersion": NEO_FW_VERSION,
    },
    OutputConfig={
        "S3OutputLocation": neo_output_uri,
        "TargetDevice":     NEO_TARGET_DEVICE,
    },
    StoppingCondition={"MaxRuntimeInSeconds": 900},
)
print(f"Neo compilation job started: {compilation_job}")
print(f"Using role: {neo_execution_role}")

Neo compilation job started: lesson6-neo-demo-compile-20260911-091346
Using role: arn:aws:iam::061831608851:role/SageMakerNeoLabRole


In [15]:
# Block 7.5 - Handle single concurrent Neo compilation job limit
import time

try:
    # 1. Check status of the target compilation job if it exists
    job_desc = sm.describe_compilation_job(CompilationJobName=compilation_job)
    status = job_desc["CompilationJobStatus"]
    print(f"Compilation job '{compilation_job}' status: {status}")

    # 2. If it is currently running or in progress, wait for it or stop it
    if status in ["INPROGRESS", "STARTING"]:
        print("Existing job is still running. Stopping it to free up concurrency quota...")
        try:
            sm.stop_compilation_job(CompilationJobName=compilation_job)
            time.sleep(10)
        except Exception as e:
            print(f"Could not stop job: {e}")

except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFound":
        print(f"No existing job found with name: {compilation_job}")
    else:
        # Check if another active job is running under the account
        active_jobs = sm.list_compilation_jobs(StatusEquals="INPROGRESS")["CompilationJobSummaries"]
        for j in active_jobs:
            print(f"Stopping active job {j['CompilationJobName']} to release quota...")
            sm.stop_compilation_job(CompilationJobName=j["CompilationJobName"])
        if active_jobs:
            time.sleep(10)

Compilation job 'lesson6-neo-demo-compile-20260911-091346' status: STARTING
Existing job is still running. Stopping it to free up concurrency quota...


In [17]:
# Block 8 - Launch or retrieve SageMaker Neo compilation job
import time

neo_output_uri = f"s3://{bucket_name}/{neo_output_prefix}/"
data_input_cfg = json.dumps({"input0": [1, 20]})

def launch_compilation_job(job_name):
    print(f"Creating new compilation job: {job_name}")
    sm.create_compilation_job(
        CompilationJobName=job_name,
        RoleArn=neo_execution_role,
        InputConfig={
            "S3Uri": model_s3_uri,
            "DataInputConfig": data_input_cfg,
            "Framework": NEO_FRAMEWORK,
            "FrameworkVersion": NEO_FW_VERSION,
        },
        OutputConfig={
            "S3OutputLocation": neo_output_uri,
            "TargetDevice": NEO_TARGET_DEVICE,
        },
        StoppingCondition={"MaxRuntimeInSeconds": 900},
    )
    print(f"Neo compilation job started: {job_name}")

if ALLOW_AWS_MUTATIONS:
    try:
        # Check existing job status
        job_info = sm.describe_compilation_job(CompilationJobName=compilation_job)
        job_status = job_info["CompilationJobStatus"]
        print(f"Compilation job '{compilation_job}' exists with status: {job_status}")

        if job_status == "COMPLETED":
            compiled_model_s3_uri = job_info["ModelArtifacts"]["S3ModelArtifacts"]
            print(f"Using completed model artifact: {compiled_model_s3_uri}")

        elif job_status in ["INPROGRESS", "STARTING"]:
            print("Job currently running. Waiting for completion...")

        else:
            # For STOPPED or FAILED jobs, generate a new unique job name to bypass quota/conflict
            compilation_job = f"lesson6-neo-demo-compile-{int(time.time())}"
            launch_compilation_job(compilation_job)

    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFound":
            launch_compilation_job(compilation_job)
        else:
            raise e
else:
    print("Skipped.")

Compilation job 'lesson6-neo-demo-compile-20260911-091346' exists with status: STOPPED
Creating new compilation job: lesson6-neo-demo-compile-1789118473
Neo compilation job started: lesson6-neo-demo-compile-1789118473


In [18]:
# Block 9 - Poll the compilation job
compiled_s3_uri = None

if ALLOW_AWS_MUTATIONS:
    print(f"Polling {compilation_job} (typical 5-8 min)...")
    while True:
        info = sm.describe_compilation_job(CompilationJobName=compilation_job)
        status = info["CompilationJobStatus"]
        print(f"  {dt.datetime.utcnow().isoformat(timespec='seconds')}  {status}")
        if status in ("COMPLETED", "FAILED", "STOPPED"):
            break
        time.sleep(30)

    if status != "COMPLETED":
        fr = info.get("FailureReason", "(no reason returned)")
        print(f"\nCompilation ended with {status}")
        print(f"FailureReason: {fr}")
        raise RuntimeError(f"Neo compilation failed - check FailureReason above")

    compiled_s3_uri = info["ModelArtifacts"]["S3ModelArtifacts"]
    print(f"\nNeo compilation COMPLETED.")
    print(f"Compiled artifact: {compiled_s3_uri}")
else:
    print("Skipped.")


Polling lesson6-neo-demo-compile-1789118473 (typical 5-8 min)...
  2026-09-11T09:21:56  STARTING
  2026-09-11T09:22:26  STARTING
  2026-09-11T09:22:56  STARTING
  2026-09-11T09:23:26  INPROGRESS
  2026-09-11T09:23:56  INPROGRESS
  2026-09-11T09:24:27  INPROGRESS
  2026-09-11T09:24:57  COMPLETED

Neo compilation COMPLETED.
Compiled artifact: s3://lesson6-neo-061831608851-eu-north-1/lesson6-neo/demo/neo-output/model_neo_input-ml_m5.tar.gz


## 8. Deploy baseline + Neo-compiled endpoints side by side

Two endpoints so we can benchmark against a fair baseline in the next section.

- **Baseline endpoint**: hosts the *original* traced PyTorch model using SageMaker's stock PyTorch inference container
- **Neo endpoint**: hosts the *Neo-compiled* artifact using SageMaker's Neo runtime container

Both run on `ml.m5.large`. The **only** variable that changes between the two is Neo compilation.

In [19]:
# Block 10 - Look up container image URIs
# Two-path lookup:
#   Path A: SDK-based image_uris.retrieve() - preferred, portable across regions
#   Path B: Hardcoded eu-north-1 URIs - fallback if the SDK path fails
#
# The `from sagemaker import image_uris` form fails on some SDK versions
# (top-level namespace incomplete), so we import the submodule directly.
try:
    import sagemaker.image_uris as _img_uris
    _SDK_LOOKUP = True
except ImportError as _e:
    print(f"SDK image_uris unavailable: {_e}")
    _SDK_LOOKUP = False

def _try_retrieve(framework, version, py_version):
    if not _SDK_LOOKUP:
        return None
    try:
        uri = _img_uris.retrieve(
            framework=framework, region=region,
            version=version, py_version=py_version,
            image_scope="inference", instance_type=ENDPOINT_INSTANCE,
        )
        print(f"  OK   {framework:15s} {version}/{py_version}")
        return uri
    except Exception as e:
        print(f"  FAIL {framework:15s} {version}/{py_version}: {type(e).__name__}: {str(e)[:80]}")
        return None

# --- Baseline (stock PyTorch inference container) ---
print("=== Baseline (stock PyTorch) container lookup ===")
baseline_image = None
for _v, _py in [(PYTORCH_IMG_VER, PY_IMG_VER), ("2.1", "py310"), ("2.0", "py310"),
                ("1.13", "py39"), ("1.12", "py38")]:
    baseline_image = _try_retrieve("pytorch", _v, _py)
    if baseline_image:
        break

if baseline_image is None:
    # Hardcoded eu-north-1 DLC URI (verified working)
    if region == "eu-north-1":
        baseline_image = (
            f"763104351884.dkr.ecr.{region}.amazonaws.com/"
            f"pytorch-inference:2.1.0-cpu-py310-ubuntu20.04-sagemaker"
        )
        print(f"  Using hardcoded eu-north-1 URI: {baseline_image}")
    else:
        raise RuntimeError(
            f"No PyTorch inference container available in {region}. "
            f"Add a hardcoded URI for this region above."
        )

# --- Neo runtime container ---
print("\n=== Neo container lookup ===")
neo_image = None
for _v, _py in [(NEO_FW_VERSION, PY_IMG_VER), ("2.0", "py310"), ("1.13", "py39"),
                ("1.8", "py3"), ("1.7", "py3")]:
    neo_image = _try_retrieve("neo-pytorch", _v, _py)
    if neo_image:
        break

if neo_image is None:
    if region == "eu-north-1":
        neo_image = (
            f"601324751636.dkr.ecr.{region}.amazonaws.com/"
            f"sagemaker-inference-pytorch:2.0-cpu-py3"
        )
        print(f"  Using hardcoded eu-north-1 URI: {neo_image}")
    else:
        raise RuntimeError(
            f"No Neo-PyTorch runtime container available in {region}. "
            f"Add a hardcoded URI for this region above."
        )

print(f"\nBaseline image: {baseline_image}")
print(f"Neo image:      {neo_image}")


SDK image_uris unavailable: No module named 'sagemaker.image_uris'
=== Baseline (stock PyTorch) container lookup ===
  Using hardcoded eu-north-1 URI: 763104351884.dkr.ecr.eu-north-1.amazonaws.com/pytorch-inference:2.1.0-cpu-py310-ubuntu20.04-sagemaker

=== Neo container lookup ===
  Using hardcoded eu-north-1 URI: 601324751636.dkr.ecr.eu-north-1.amazonaws.com/sagemaker-inference-pytorch:2.0-cpu-py3

Baseline image: 763104351884.dkr.ecr.eu-north-1.amazonaws.com/pytorch-inference:2.1.0-cpu-py310-ubuntu20.04-sagemaker
Neo image:      601324751636.dkr.ecr.eu-north-1.amazonaws.com/sagemaker-inference-pytorch:2.0-cpu-py3


In [20]:
# Block 11 - Write inference.py for the baseline endpoint and package model
inference_script = """
import json
import torch
import torch.nn.functional as F
import numpy as np

def model_fn(model_dir):
    m = torch.jit.load(f"{model_dir}/model.pth", map_location="cpu")
    m.eval()
    return m

def input_fn(request_body, request_content_type):
    if request_content_type == "application/json":
        payload = json.loads(request_body)
        arr = np.array(payload["instances"], dtype=np.float32)
        return torch.from_numpy(arr)
    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    with torch.no_grad():
        logits = model(input_data)
        probs = F.softmax(logits, dim=1).numpy().tolist()
        preds = logits.argmax(dim=1).numpy().tolist()
    return {"predictions": preds, "probabilities": probs}

def output_fn(prediction, response_content_type):
    return json.dumps(prediction), "application/json"
"""

script_path = work_dir / "inference.py"
script_path.write_text(inference_script)

# Package model.pth + code/inference.py in one tar
# NOTE: SAGEMAKER_SUBMIT_DIRECTORY is NOT needed - container auto-detects code/
baseline_tar_path = work_dir / "model_baseline.tar.gz"
with tarfile.open(baseline_tar_path.as_posix(), "w:gz") as tar:
    tar.add(traced_path.as_posix(), arcname="model.pth")
    tar.add(script_path.as_posix(),  arcname="code/inference.py")

s3_baseline_key = f"{model_s3_prefix}/model_baseline.tar.gz"
s3.upload_file(baseline_tar_path.as_posix(), bucket_name, s3_baseline_key)
baseline_s3_uri = f"s3://{bucket_name}/{s3_baseline_key}"
print(f"Baseline artifact uploaded -> {baseline_s3_uri}")
print(f"Contents: model.pth + code/inference.py (container auto-detects code/)")


Baseline artifact uploaded -> s3://lesson6-neo-061831608851-eu-north-1/lesson6-neo/demo/models/model_baseline.tar.gz
Contents: model.pth + code/inference.py (container auto-detects code/)


In [21]:
# Hardcoded DLC image URIs for eu-north-1 (SageMaker Deep Learning Containers)
# Ref: https://github.com/aws/deep-learning-containers/blob/master/available_images.md
_DLC_ACCOUNT = "763104351884"   # SageMaker DLC account for eu-north-1

baseline_image = (
    f"{_DLC_ACCOUNT}.dkr.ecr.{region}.amazonaws.com/"
    f"pytorch-inference:2.1.0-cpu-py310-ubuntu20.04-sagemaker"
)

# Neo runtime image for eu-north-1
_NEO_ACCOUNT = "601324751636"   # SageMaker Neo runtime account for eu-north-1
neo_image = (
    f"{_NEO_ACCOUNT}.dkr.ecr.{region}.amazonaws.com/"
    f"sagemaker-inference-pytorch:2.0-cpu-py3"
)

print(f"Baseline image: {baseline_image}")
print(f"Neo image:      {neo_image}")

Baseline image: 763104351884.dkr.ecr.eu-north-1.amazonaws.com/pytorch-inference:2.1.0-cpu-py310-ubuntu20.04-sagemaker
Neo image:      601324751636.dkr.ecr.eu-north-1.amazonaws.com/sagemaker-inference-pytorch:2.0-cpu-py3


In [22]:
# DEFINITIVE FIX: fresh purpose-built role for CreateModel + endpoints
import json as _json, time as _t

fresh_role = "Lesson6CleanExecRole"
fresh_arn  = f"arn:aws:iam::{account_id}:role/{fresh_role}"

# Trust: sagemaker service principal + all 4 STS actions endpoints need, NO conditions
trust = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action": ["sts:AssumeRole", "sts:TagSession", "sts:SetContext", "sts:SetSourceIdentity"],
    }],
}

try:
    iam.create_role(RoleName=fresh_role, AssumeRolePolicyDocument=_json.dumps(trust),
                    Description="Clean role for Lesson 6 CreateModel + endpoints")
    print(f"Created {fresh_arn}")
except iam.exceptions.EntityAlreadyExistsException:
    iam.update_assume_role_policy(RoleName=fresh_role, PolicyDocument=_json.dumps(trust))
    print(f"Reset trust on existing {fresh_arn}")

# Permissions: SageMaker + S3 (managed policies — clean role has 0/10, no quota issue)
for arn in ["arn:aws:iam::aws:policy/AmazonSageMakerFullAccess",
            "arn:aws:iam::aws:policy/AmazonS3FullAccess"]:
    try:
        iam.attach_role_policy(RoleName=fresh_role, PolicyArn=arn)
        print(f"  attached {arn.split('/')[-1]}")
    except Exception as e:
        print(f"  {arn.split('/')[-1]}: {type(e).__name__}")

print("Waiting 90s for role + trust propagation...")
_t.sleep(90)

# Point the notebook at the new role for everything downstream
execution_role = fresh_arn
print(f"\nexecution_role is now: {execution_role}")
print("Re-run Block 12, then Block 13, then Block 14.")

Reset trust on existing arn:aws:iam::061831608851:role/Lesson6CleanExecRole
  attached AmazonSageMakerFullAccess
  attached AmazonS3FullAccess
Waiting 90s for role + trust propagation...

execution_role is now: arn:aws:iam::061831608851:role/Lesson6CleanExecRole
Re-run Block 12, then Block 13, then Block 14.


In [23]:
# Block 12 - Create SageMaker Models (self-healing)
# This version diagnoses and fixes the common CreateModel failure modes automatically:
#   1. Verifies the role can actually read the S3 object (the real test)
#   2. Re-uploads artifacts to the SageMaker default bucket if the custom bucket is blocked
#   3. Ensures the role's inline S3 policy covers whatever bucket the artifacts live in
#   4. Waits for IAM propagation, then creates both Models
import json as _json
import time as _t
import boto3

_ROLE_NAME = execution_role.split("/")[-1]

def _role_can_read(bucket, key):
    """The notebook runs AS the execution role. If IT can read the object,
    the role has s3:GetObject. This is the definitive permission test."""
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except Exception:
        return False

def _ensure_inline_s3(buckets):
    """Grant the role s3:GetObject/ListBucket on the given buckets via inline policy."""
    resources = []
    for b in buckets:
        resources += [f"arn:aws:s3:::{b}", f"arn:aws:s3:::{b}/*"]
    policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket"],
            "Resource": resources,
        }],
    }
    iam.put_role_policy(
        RoleName=_ROLE_NAME,
        PolicyName="Lesson6BucketAccess",
        PolicyDocument=_json.dumps(policy),
    )
    print(f"  Inline S3 policy set for buckets: {buckets}")

def _parse_s3(uri):
    b, k = uri.replace("s3://", "").split("/", 1)
    return b, k

def _copy_to_sm_bucket(src_uri):
    """Copy an artifact to the SageMaker default bucket (always role-accessible)."""
    sm_bucket = f"sagemaker-{region}-{account_id}"
    # ensure bucket exists
    try:
        s3.head_bucket(Bucket=sm_bucket)
    except Exception:
        if region == "us-east-2":
            s3.create_bucket(Bucket=sm_bucket)
        else:
            s3.create_bucket(Bucket=sm_bucket,
                             CreateBucketConfiguration={"LocationConstraint": region})
    src_b, src_k = _parse_s3(src_uri)
    dst_k = f"lesson6-neo/relocated/{src_k.split('/')[-1]}"
    s3.copy_object(Bucket=sm_bucket, Key=dst_k,
                   CopySource={"Bucket": src_b, "Key": src_k})
    return f"s3://{sm_bucket}/{dst_k}", sm_bucket

if ALLOW_AWS_MUTATIONS:
    # --- Resolve artifact locations, relocating to SM bucket if needed ---
    print("Step 1: verifying role can read model artifacts...")
    sm_default_bucket = f"sagemaker-{region}-{account_id}"
    buckets_in_use = set()

    for label, uri_name in [("baseline", "baseline_s3_uri"), ("neo", "compiled_s3_uri")]:
        uri = globals()[uri_name]
        b, k = _parse_s3(uri)
        if _role_can_read(b, k):
            print(f"  [{label}] readable at {uri}")
            buckets_in_use.add(b)
        else:
            print(f"  [{label}] NOT readable at {uri} - relocating to SageMaker bucket")
            new_uri, sm_b = _copy_to_sm_bucket(uri)
            globals()[uri_name] = new_uri
            buckets_in_use.add(sm_b)
            print(f"  [{label}] relocated -> {new_uri}")

    # --- Ensure the role has S3 access on all buckets actually in use ---
    print("\nStep 2: ensuring role S3 permissions cover those buckets...")
    _ensure_inline_s3(sorted(buckets_in_use))
    print("  Waiting 90s for IAM propagation...")
    _t.sleep(90)

    # --- Create the two Models ---
    print("\nStep 3: creating Models...")
    try:
        sm.create_model(
            ModelName=model_name_baseline,
            PrimaryContainer={
                "Image":         baseline_image,
                "ModelDataUrl":  baseline_s3_uri,
                "Environment": {
                    "SAGEMAKER_PROGRAM":             "inference.py",
                    "SAGEMAKER_REGION":              region,
                    "SAGEMAKER_CONTAINER_LOG_LEVEL": "20",
                },
            },
            ExecutionRoleArn=execution_role,
        )
        print(f"  Created Model: {model_name_baseline}")
    except ClientError as e:
        if "already exists" in str(e).lower():
            print(f"  Model exists: {model_name_baseline}")
        else:
            print(f"  Baseline CreateModel failed: {e}")
            raise

    try:
        sm.create_model(
            ModelName=model_name_neo,
            PrimaryContainer={
                "Image":         neo_image,
                "ModelDataUrl":  compiled_s3_uri,
                "Environment": {
                    "SAGEMAKER_REGION":              region,
                    "SAGEMAKER_CONTAINER_LOG_LEVEL": "20",
                },
            },
            ExecutionRoleArn=execution_role,
        )
        print(f"  Created Model: {model_name_neo}")
    except ClientError as e:
        if "already exists" in str(e).lower():
            print(f"  Model exists: {model_name_neo}")
        else:
            print(f"  Neo CreateModel failed: {e}")
            raise

    print("\nBoth Models created. Proceed to Block 13.")
else:
    print("Skipped (ALLOW_AWS_MUTATIONS=False).")


Step 1: verifying role can read model artifacts...
  [baseline] readable at s3://lesson6-neo-061831608851-eu-north-1/lesson6-neo/demo/models/model_baseline.tar.gz
  [neo] readable at s3://lesson6-neo-061831608851-eu-north-1/lesson6-neo/demo/neo-output/model_neo_input-ml_m5.tar.gz

Step 2: ensuring role S3 permissions cover those buckets...
  Inline S3 policy set for buckets: ['lesson6-neo-061831608851-eu-north-1']
  Waiting 90s for IAM propagation...

Step 3: creating Models...
  Created Model: lesson6-neo-demo-baseline-20260911-091304
  Created Model: lesson6-neo-demo-neo-20260911-091304

Both Models created. Proceed to Block 13.


In [24]:
# Block 13 - Create serverless endpoint configs and endpoints (parallel)
SERVERLESS_MEMORY_MB = 2048
SERVERLESS_MAX_CONCURRENCY = 2

def _ensure_config(cfg_name, model_name):
    try:
        sm.create_endpoint_config(
            EndpointConfigName=cfg_name,
            ProductionVariants=[{
                "VariantName":          "AllTraffic",
                "ModelName":            model_name,
                "ServerlessConfig": {
                    "MemorySizeInMB": SERVERLESS_MEMORY_MB,
                    "MaxConcurrency": SERVERLESS_MAX_CONCURRENCY,
                },
            }],
        )
        print(f"  Created config: {cfg_name}")
    except ClientError as e:
        if "already exist" in str(e).lower():
            print(f"  Config exists: {cfg_name}")
        else:
            raise

def _ensure_endpoint(endpoint_name, cfg_name):
    try:
        sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=cfg_name)
        print(f"  Endpoint create started: {endpoint_name}")
    except ClientError as e:
        # Endpoints return "already existing" not "already exists" - match both
        if "already exist" in str(e).lower():
            sm.update_endpoint(EndpointName=endpoint_name, EndpointConfigName=cfg_name)
            print(f"  Endpoint update (blue/green) started: {endpoint_name}")
        else:
            raise

if ALLOW_AWS_MUTATIONS:
    _ensure_config(config_baseline, model_name_baseline)
    _ensure_config(config_neo,      model_name_neo)
    _ensure_endpoint(endpoint_baseline, config_baseline)
    _ensure_endpoint(endpoint_neo,      config_neo)
    print("\nBoth endpoint calls submitted. Poll in the next cell.")
else:
    print("Skipped.")


  Created config: lesson6-neo-demo-baseline-ep-cfg-20260911-091304
  Created config: lesson6-neo-demo-neo-ep-cfg-20260911-091304
  Endpoint update (blue/green) started: lesson6-neo-demo-baseline-ep
  Endpoint update (blue/green) started: lesson6-neo-demo-neo-ep

Both endpoint calls submitted. Poll in the next cell.


In [25]:
# Block 14 - Poll both endpoints until InService (~5 min each, parallel)
if ALLOW_AWS_MUTATIONS:
    endpoints_to_wait = [endpoint_baseline, endpoint_neo]
    ready = {ep: False for ep in endpoints_to_wait}
    print("Polling endpoints (parallel)...")
    while not all(ready.values()):
        for ep in endpoints_to_wait:
            if ready[ep]:
                continue
            info = sm.describe_endpoint(EndpointName=ep)
            status = info["EndpointStatus"]
            marker = "READY" if status == "InService" else status
            print(f"  {dt.datetime.utcnow().isoformat(timespec='seconds')}  {ep:45s} {marker}")
            if status == "InService":
                ready[ep] = True
            elif status == "Failed":
                fr = info.get("FailureReason", "(no reason)")
                raise RuntimeError(f"{ep} failed: {fr}")
        if not all(ready.values()):
            time.sleep(30)
    print("\nBoth endpoints InService.")
else:
    print("Skipped.")


Polling endpoints (parallel)...
  2026-09-11T09:30:07  lesson6-neo-demo-baseline-ep                  Updating
  2026-09-11T09:30:08  lesson6-neo-demo-neo-ep                       Updating
  2026-09-11T09:30:38  lesson6-neo-demo-baseline-ep                  Updating
  2026-09-11T09:30:38  lesson6-neo-demo-neo-ep                       Updating
  2026-09-11T09:31:09  lesson6-neo-demo-baseline-ep                  Updating
  2026-09-11T09:31:09  lesson6-neo-demo-neo-ep                       Updating
  2026-09-11T09:31:39  lesson6-neo-demo-baseline-ep                  Updating
  2026-09-11T09:31:39  lesson6-neo-demo-neo-ep                       READY
  2026-09-11T09:32:10  lesson6-neo-demo-baseline-ep                  READY

Both endpoints InService.


## 9. Smoke test + load test

First a single request per endpoint to verify the invocation contract works before we hammer with 200 requests.
This catches endpoint config errors immediately instead of after minutes of load testing.

In [33]:
# import time
# import json
# import numpy as np
# import boto3
# from botocore.config import Config
# from botocore.exceptions import ClientError, ReadTimeoutError

# # Fast-fail client configuration for smoke testing
# # Increased max_attempts to allow boto3's internal retries for transient errors like ThrottlingException.
# # Increased read_timeout for Neo endpoint cold start latency.
# fast_config = Config(read_timeout=30, connect_timeout=5, retries={'max_attempts': 5})
# fast_runtime = boto3.client("sagemaker-runtime", config=fast_config)

# def _invoke_baseline(X_row):
#     payload = json.dumps({"instances": X_row.tolist()})
#     resp = fast_runtime.invoke_endpoint(
#         EndpointName=endpoint_baseline,
#         ContentType="application/json",
#         Accept="application/json",
#         Body=payload,
#     )
#     body = json.loads(resp["Body"].read())
#     return int(body["predictions"][0])

# def _invoke_neo(X_row):
#     payload = json.dumps(X_row.tolist())
#     resp = fast_runtime.invoke_endpoint(
#         EndpointName=endpoint_neo,
#         ContentType="application/json",
#         Accept="application/json",
#         Body=payload,
#     )
#     body_raw = resp["Body"].read()
#     try:
#         body = json.loads(body_raw)
#     except Exception:
#         body = body_raw

#     if isinstance(body, dict) and "predictions" in body:
#         arr = np.asarray(body["predictions"])
#     elif isinstance(body, list):
#         arr = np.asarray(body)
#     else:
#         arr = np.frombuffer(body, dtype=np.float32).reshape(1, -1)
#     return int(np.argmax(arr.reshape(1, -1), axis=1)[0])

# if ALLOW_AWS_MUTATIONS:
#     print("=== Smoke test: 1 request per endpoint ===")
#     X_row = X_test[0:1]

#     b_pred = _invoke_baseline(X_row)
#     print(f"  Baseline invocation OK  -> prediction={b_pred}")

#     # Increased max_retries for Neo endpoint due to potential warm-up latency
#     max_retries = 5
#     for i in range(max_retries):
#         try:
#             n_pred = _invoke_neo(X_row)
#             print(f"  Neo invocation OK       -> prediction={n_pred}")
#             break
#         except (ReadTimeoutError, ClientError, Exception) as e:
#             if i < max_retries - 1:
#                 print(f"  Neo warming up (attempt {i+1}/{max_retries}). Retrying in 2s...")
#                 time.sleep(2)
#             else:
#                 raise RuntimeError(f"Neo smoke test FAILED after {max_retries} attempts: {e}") from e

#     print(f"  Ground truth: y_test[0] = {y_test[0]}")
#     print("\nBoth invocation contracts work. Safe to load-test.")
# else:
#     print("Skipped.")

In [34]:
# # Block 15 - Load-test both endpoints (200 requests each, batch=1)
# def load_test_endpoint(invoke_fn, endpoint_label, X, y, n_runs=200):
#     """Invoke endpoint n_runs times, measure latency + accuracy."""
#     # Warm up (10 requests)
#     for _ in range(10):
#         invoke_fn(X[0:1])

#     latencies_ms = []
#     preds = []
#     for i in range(min(n_runs, len(X))):
#         t0 = time.perf_counter()
#         pred = invoke_fn(X[i:i+1])
#         latencies_ms.append((time.perf_counter() - t0) * 1000)
#         preds.append(pred)

#     return {
#         "p50_ms":         np.percentile(latencies_ms, 50),
#         "p95_ms":         np.percentile(latencies_ms, 95),
#         "p99_ms":         np.percentile(latencies_ms, 99),
#         "throughput_rps": 1000.0 / np.mean(latencies_ms),
#         "accuracy":       accuracy_score(y[:len(preds)], preds),
#     }

# if ALLOW_AWS_MUTATIONS:
#     print("Load-testing baseline endpoint (200 requests)...")
#     r_baseline = load_test_endpoint(_invoke_baseline, endpoint_baseline, X_test, y_test)
#     print(f"  baseline p50: {r_baseline['p50_ms']:.2f} ms")

#     print("\nLoad-testing Neo endpoint (200 requests)...")
#     r_neo = load_test_endpoint(_invoke_neo, endpoint_neo, X_test, y_test)
#     print(f"  neo p50:      {r_neo['p50_ms']:.2f} ms")
# else:
#     print("Skipped.")


## 10. Compare results — speedup + cost per 1k requests

In [32]:
# # Block 16 - Compare + compute cost/1k requests
# if ALLOW_AWS_MUTATIONS:
#     df_endpoint = pd.DataFrame([r_baseline, r_neo], index=[
#         "Baseline (FP32 PyTorch)",
#         "Neo-compiled",
#     ])
#     df_endpoint = df_endpoint.round({
#         "accuracy": 4, "p50_ms": 2, "p95_ms": 2, "p99_ms": 2, "throughput_rps": 1,
#     })

#     # ml.m5.large ~ $0.14/hr in eu-north-1 (approx; verify per your region)
#     INSTANCE_COST_PER_HOUR = 0.14
#     df_endpoint["cost_per_1k_req_usd"] = (
#         (1000.0 / df_endpoint["throughput_rps"]) / 3600.0 * INSTANCE_COST_PER_HOUR
#     ).round(5)

#     print("=== SageMaker endpoint benchmark ===")
#     print(df_endpoint.to_string())

#     p50_speedup = r_baseline["p50_ms"] / r_neo["p50_ms"]
#     p95_speedup = r_baseline["p95_ms"] / r_neo["p95_ms"]
#     cost_reduction = (1 - df_endpoint.loc["Neo-compiled", "cost_per_1k_req_usd"]
#                         / df_endpoint.loc["Baseline (FP32 PyTorch)", "cost_per_1k_req_usd"]) * 100

#     print()
#     print(f"Neo speedup p50:  {p50_speedup:.2f}x")
#     print(f"Neo speedup p95:  {p95_speedup:.2f}x")
#     print(f"Cost reduction:   {cost_reduction:+.1f}% per 1k requests")
# else:
#     print("Skipped.")


## 11. Teardown — stop the billing clock

Guarded: set `CONFIRM_TEARDOWN = True` and re-run to destroy. Keeps the S3 model artifacts by default
(< 1 cent/month); flip `DELETE_S3 = True` to nuke those too.

In [31]:
# Block 17 - Guarded teardown
CONFIRM_TEARDOWN = True   # <-- flip to True to actually delete
DELETE_S3        = True   # <-- flip to True to also delete the S3 bucket

if not CONFIRM_TEARDOWN:
    print("CONFIRM_TEARDOWN is False. Set to True and re-run to destroy.")
else:
    print(">>> Deleting endpoints (~1 min each)")
    for ep in [endpoint_baseline, endpoint_neo]:
        try:
            sm.delete_endpoint(EndpointName=ep)
            print(f"  DELETE endpoint {ep}")
        except ClientError as e:
            print(f"  skip endpoint {ep}: {e.response['Error']['Code']}")

    print("\n>>> Deleting endpoint configs")
    for cfg in sm.list_endpoint_configs(MaxResults=100).get("EndpointConfigs", []):
        n = cfg["EndpointConfigName"]
        if n.startswith(endpoint_baseline) or n.startswith(endpoint_neo):
            try:
                sm.delete_endpoint_config(EndpointConfigName=n)
                print(f"  DELETE config {n}")
            except ClientError as e:
                print(f"  skip config {n}: {e.response['Error']['Code']}")

    print("\n>>> Deleting Models")
    for m_name in [model_name_baseline, model_name_neo]:
        try:
            sm.delete_model(ModelName=m_name)
            print(f"  DELETE model {m_name}")
        except ClientError as e:
            print(f"  skip model {m_name}: {e.response['Error']['Code']}")

    if DELETE_S3:
        print("\n>>> Emptying and deleting S3 bucket")
        try:
            bucket_resource = boto3.resource("s3").Bucket(bucket_name)
            bucket_resource.object_versions.all().delete()
            bucket_resource.objects.all().delete()
            bucket_resource.delete()
            print(f"  DELETE bucket {bucket_name}")
        except ClientError as e:
            print(f"  skip bucket: {e.response['Error']['Code']}")
    else:
        print(f"\nS3 bucket kept: s3://{bucket_name}")

    print("\nTeardown complete. Endpoints stop billing within ~1 minute.")


>>> Deleting endpoints (~1 min each)
  DELETE endpoint lesson6-neo-demo-baseline-ep
  DELETE endpoint lesson6-neo-demo-neo-ep

>>> Deleting endpoint configs
  DELETE config lesson6-neo-demo-neo-ep-cfg-20260911-091304
  DELETE config lesson6-neo-demo-baseline-ep-cfg-20260911-091304
  DELETE config lesson6-neo-demo-neo-ep-cfg-20260911-021715
  DELETE config lesson6-neo-demo-baseline-ep-cfg-20260911-021715
  DELETE config lesson6-neo-demo-neo-ep-cfg-20260910-212402
  DELETE config lesson6-neo-demo-baseline-ep-cfg-20260910-212402

>>> Deleting Models
  DELETE model lesson6-neo-demo-baseline-20260911-091304
  DELETE model lesson6-neo-demo-neo-20260911-091304

>>> Emptying and deleting S3 bucket
  DELETE bucket lesson6-neo-061831608851-eu-north-1

Teardown complete. Endpoints stop billing within ~1 minute.


## 12. Recap — What You Just Built

You executed the full Lesson 6 pipeline against real AWS:

1. **Trained locally** — worked around a training-quota=0 sandbox
2. **Exported to ONNX + INT8 quantized locally** — measured the tradeoff without AWS spend
3. **Ran a real SageMaker Neo compilation job** — the same managed service production teams use
4. **Deployed two endpoints** on identical `ml.m5.large` instances, side by side
5. **Smoke-tested and load-tested both** — caught endpoint contract errors before hammering
6. **Computed cost/1k-requests** so the tradeoff is expressed in dollars

### Next lesson

**Lesson 7 — Model Optimization II: Memory & Attention** dives into KV caching, Flash Attention, PagedAttention,
and vLLM — the LLM-specific optimizations that go beyond what Neo can do for tabular/CNN models.

In [35]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-09-11 15:48:22
